<a href="https://colab.research.google.com/github/Dev1ze32/llma3.2_3b_model_test/blob/main/content/02-getting-started/jupyter_notebooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLAMA 3.2 3B MODEL TESTING

In [ ]:
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

# 1. This triggers an interactive input box.
# Paste your token here when prompted and hit enter.
login()

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Configure 4-bit quantization to fit safely within Colab's free T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model (this will take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Initialize the text generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Format your prompt using Llama 3.2's chat template structure
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a quick Python function to check if a word is a palindrome."}
]

print("Running inference...\n")
outputs = pipe(messages, max_new_tokens=512)

print("Result:")
print(outputs[0]["generated_text"][-1]["content"])

# EMBEDDING MODEL TESTING

In [ ]:
import os
import torch
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer, util

def extract_and_chunk_pdf(file_path, chunk_size=300):
    """
    Reads a PDF and splits it into smaller chunks of text.
    Chunking is necessary because models have a max token limit (512 for E5-base).
    """
    print(f"Reading {file_path}...")
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + " "

    # Simple word-based chunking
    words = text.split()
    chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

def main():
    pdf_path = "handbook.pdf"

    if not os.path.exists(pdf_path):
        print(f"Error: {pdf_path} not found in the current directory.")
        return

    # 1. Load Knowledge Base
    raw_passages = extract_and_chunk_pdf(pdf_path, chunk_size=300)

    # 2. Load the E5 Model
    print("Loading intfloat/multilingual-e5-base...")
    model = SentenceTransformer('intfloat/multilingual-e5-base')

    # 3. Format and Encode Passages
    # E5 REQUIREMENT: Documents must be prefixed with "passage: "
    formatted_passages = [f"passage: {text}" for text in raw_passages]

    print(f"Encoding {len(formatted_passages)} chunks into vector embeddings...")
    # normalize_embeddings=True is recommended for cosine similarity
    passage_embeddings = model.encode(
        formatted_passages,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    print("\nKnowledge Base Ready!\n")

    # 4. Interactive Search Loop
    while True:
        user_query = input("Ask a question about the handbook (or type 'quit'): ")
        if user_query.lower() in ['quit', 'exit', 'q']:
            break

        # E5 REQUIREMENT: Search queries must be prefixed with "query: "
        formatted_query = f"query: {user_query}"

        # Encode the query
        query_embedding = model.encode(
            formatted_query,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        # 5. Calculate Cosine Similarity to find the best matches
        hits = util.semantic_search(query_embedding, passage_embeddings, top_k=3)[0]

        print("\n" + "="*50)
        print("TOP 3 SEARCH RESULTS:")
        print("="*50)
        for i, hit in enumerate(hits, 1):
            score = hit['score']
            chunk_text = raw_passages[hit['corpus_id']]
            print(f"\n--- Result {i} (Confidence Score: {score:.4f}) ---")
            print(f"{chunk_text}...\n")

if __name__ == "__main__":
    main()

# LLAMA 3.2 3B AND EMBEDDING MODEL COMBINED

In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub sentence-transformers PyPDF2 torch

In [ ]:
import os
import torch
from huggingface_hub import login
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

# ==========================================
# 1. AUTHENTICATION
# ==========================================
# This triggers the interactive login box. Paste your HF token here.
login()

# ==========================================
# 2. CONFIGURATION & MODULAR PROMPTS
# ==========================================
CONFIG = {
    "pdf_path": "handbook.pdf",
    "embedding_model_id": "intfloat/multilingual-e5-base",
    "llm_model_id": "meta-llama/Llama-3.2-3B-Instruct",
    "chunk_size": 300,        # Word count per chunk
    "top_k_passages": 3,      # Number of document fragments retrieved
    "max_new_tokens": 512,    # Maximum output length from Llama
}

# Modifying these variables allows you to change the system instructions effortlessly
PROMPT_TEMPLATES = {
    "SYSTEM_PROMPT": (
        "You are a helpful, precise institutional assistant. Your task is to answer "
        "the user's questions relying strictly on the provided handbook context below.\n"
        "If the answer cannot be found in the context, politely state that you do "
        "not know based on the available documentation. Do not invent facts."
    ),
    "USER_CONTEXT_WRAPPER": (
        "Context from the handbook:\n"
        "---------------------\n"
        "{retrieved_context}\n"
        "---------------------\n\n"
        "User Question: {user_query}"
    )
}

# ==========================================
# 3. HELPER FUNCTIONS FOR PDF PROCESSING
# ==========================================
def extract_and_chunk_pdf(file_path, chunk_size=300):
    """Extracts text from a PDF and chunks it into smaller blocks."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Could not find the file: {file_path}. Please upload it to Colab.")

    print(f"[{file_path}] Extracting text contents...")
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + " "

    words = text.split()
    chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    print(f"Successfully processed PDF into {len(chunks)} text chunks.")
    return chunks

# ==========================================
# 4. INITIALIZATION ENGINE
# ==========================================
print("\n--- Initializing Knowledge Base & Embedding Model ---")
# Extract text chunks
raw_passages = extract_and_chunk_pdf(CONFIG["pdf_path"], chunk_size=CONFIG["chunk_size"])

# Format text chunks with E5 prefix requirement
formatted_passages = [f"passage: {text}" for text in raw_passages]

# Initialize E5 Embedding Model
embed_model = SentenceTransformer(CONFIG["embedding_model_id"])

print("Generating vector embeddings for the handbook context...")
passage_embeddings = embed_model.encode(
    formatted_passages,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("\n--- Initializing Llama 3.2 3B Generation Model ---")
# Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading LLM tokenizer...")
llm_tokenizer = AutoTokenizer.from_pretrained(CONFIG["llm_model_id"])

print("Loading LLM model weights (this might take a minute)...")
llm_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["llm_model_id"],
    quantization_config=bnb_config,
    device_map="auto"
)

# Initialize generation pipeline
llm_pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer
)

print("\n Setup Complete! The RAG Assistant is ready to accept questions.")

# ==========================================
# 5. CORE INTERACTION PIPELINE
# ==========================================
def ask_assistant(query: str):
    """
    RAG Execution Flow:
    1. Embeds the user query with the 'query: ' prefix.
    2. Performs semantic search across document embeddings.
    3. Assembles the system instructions and contextual user payload.
    4. Generates an informed response via Llama 3.2.
    """
    # Step A: Embed Query (E5 requirement)
    formatted_query = f"query: {query}"
    query_embedding = embed_model.encode(
        formatted_query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    # Step B: Match Similar Vectors
    hits = util.semantic_search(query_embedding, passage_embeddings, top_k=CONFIG["top_k_passages"])[0]

    # Step C: Consolidate Top Context Passages
    retrieved_chunks = [raw_passages[hit['corpus_id']] for hit in hits]
    combined_context = "\n\n".join(retrieved_chunks)

    # Step D: Construct Modular Prompts
    system_prompt = PROMPT_TEMPLATES["SYSTEM_PROMPT"]
    user_content = PROMPT_TEMPLATES["USER_CONTEXT_WRAPPER"].format(
        retrieved_context=combined_context,
        user_query=query
    )

    # Step E: Format structured payload for Llama 3.2's template chat architecture
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]

    # Step F: Run Local Inference
    outputs = llm_pipe(messages, max_new_tokens=CONFIG["max_new_tokens"])
    return outputs[0]["generated_text"][-1]["content"]

# ==========================================
# 6. RUN AN INTERACTIVE LOOP
# ==========================================
while True:
    user_input = input("\nEnter your question (or type 'exit' to quit): ")
    if user_input.lower() in ['exit', 'quit', 'q']:
        print("Shutting down the assistant session.")
        break

    if not user_input.strip():
        continue

    print("\nThinking...")
    response = ask_assistant(user_input)
    print(f"\n[Assistant Response]:\n{response}")
    print("-" * 40)